### **1. Introduction & Recap**

*   **Problem:** Large Language Models (LLMs) naturally return **unstructured textual responses**. This makes it difficult to integrate them directly with other systems like databases or APIs that require structured data.
*   **Recap from Previous Video (Structured Output):**
    *   **Goal:** Force the LLM to return a structured response (like JSON) instead of plain text.
    *   **Two Types of LLMs:**
        1.  **Models that can natively give structured output** (e.g., newer GPT models). In LangChain, you can use the `with_structured_output` method for these.
        2.  **Models that cannot natively give structured output** (e.g., many open-source models like TinyLlama). These require manual guidance.
*   **Solution for Today's Video:** **Output Parsers**. They help convert the raw, unstructured text from *any* LLM into a desired structured format.

### **2. What are Output Parsers?**

*   **Definition:** Classes in LangChain that help convert raw LLM responses (text) into structured formats like **JSON, CSV, Pydantic models**, etc.
*   **Purpose:** They ensure **consistency**, **validation**, and **ease of use** in your applications.
*   **Key Point:** They work seamlessly with **both** types of LLMs (those that can and cannot natively produce structured output).
*   **Four Main Parsers Covered:**
    1.  String Output Parser
    2.  JSON Output Parser
    3.  Structured Output Parser
    4.  Pydantic Output Parser

---



### **4. Parser 2: JSON Output Parser**

*   **Purpose:** Forces the LLM to return its output in a **JSON format**.
*   **Limitation:** It does **not enforce a specific schema**. The LLM decides the structure (keys) of the JSON. You can only hint at it in the prompt.

### 1. Initialize Model and Parser

In [1]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("Hugging_face_api_token")

# Create LLM endpoint
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)

# Wrap with chat interface
model = ChatHuggingFace(llm=llm)


In [ ]:
# --- Imports ---
from langchain_core.output_parsers import JsonOutputParser
json_parser=JsonOutputParser()

### 2. Create prompt Template with Format Instructions
###  Note the '{format_instructions}' variable.

In [6]:
from langchain_core.prompts import PromptTemplate

template=PromptTemplate(
    template="Give me the name,age and city of a fictional person \n {format_instructions}",
    input_variables=[],#No input variables here
    partial_variables={
        "format_instructions":json_parser.get_format_instructions()
    }
)
print(template)

input_variables=[] input_types={} partial_variables={'format_instructions': 'Return a JSON object.'} template='Give me the name,age and city of a fictional person \n {format_instructions}'


### 3.create and invoke chain

In [7]:
chain=template | model |json_parser
result=chain.invoke({}) # pass an empty dict as no input variables needed

### 4.Output

In [8]:
print(result)
print(type(result))

{'name': 'Emma Swan', 'age': 28, 'city': 'Storybrooke'}
<class 'dict'>


### Very Important
- Limitation: It does not enforce a specific schema.
- The LLM decides the structure (keys) of the JSON. You can only hint at it in the prompt.

In [9]:
print(result.keys())

dict_keys(['name', 'age', 'city'])
